# Notebook 01 — Fusion Data Preparation

## Goal

Prepare the multimodal dataset used by the Fusion Module.

This notebook:

- Loads image features extracted from the SegResNet encoder.
- Loads text embeddings extracted from BioBERT and Bio_ClinicalBERT.
- Verifies patient alignment across both modalities.
- Builds multimodal fusion datasets.
- Saves reusable datasets for subsequent Fusion notebooks.

The generated datasets will be used by the Fusion Encoder without
requiring access to the original CV or NLP pipelines.

# Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Configuration

In [2]:
CV_DIR = Path(
    "/kaggle/input/datasets/mariammohamed1095/workingggg/reports/results"
)

BIOBERT_DIR = Path(
    "/kaggle/input/datasets/mariammohamed1095/biobert/datasets/processed/nlp/biobert-base-cased-v1.1"
)

CLINICAL_DIR = Path(
    "/kaggle/input/datasets/mariammohamed1095/clinicalbert/datasets/processed/nlp/Bio_ClinicalBERT"
)

OUTPUT_DIR = Path(
    "/kaggle/working/datasets/processed/fusion"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [3]:
print(CV_DIR)

print(BIOBERT_DIR)

print(CLINICAL_DIR)

print(OUTPUT_DIR)

/kaggle/input/datasets/mariammohamed1095/workingggg/reports/results
/kaggle/input/datasets/mariammohamed1095/biobert/datasets/processed/nlp/biobert-base-cased-v1.1
/kaggle/input/datasets/mariammohamed1095/clinicalbert/datasets/processed/nlp/Bio_ClinicalBERT
/kaggle/working/datasets/processed/fusion


# Load Image Features

In [4]:
def load_cv_features():

    cv = {}

    for split in [

        "train",
        "validation",
        "test",

    ]:

        cv[split] = {

            "features": np.load(
                CV_DIR / f"bottleneck_features_{split}.npy"
            ),

            "metadata": pd.read_csv(
                CV_DIR / f"{split}_metadata.csv"
            ),

        }

    return cv

In [5]:
cv = load_cv_features()

In [6]:
for split in cv:

    print("=" * 60)

    print(split.upper())

    print(
        "Features :",
        cv[split]["features"].shape,
    )

    print(
        "Metadata :",
        cv[split]["metadata"].shape,
    )

TRAIN
Features : (257, 256)
Metadata : (257, 4)
VALIDATION
Features : (56, 256)
Metadata : (56, 4)
TEST
Features : (56, 256)
Metadata : (56, 4)


# Load Text Embeddings

In [7]:
def load_nlp_embeddings(folder):

    nlp = {}

    for split in [

        "train",
        "validation",
        "test",

    ]:

        nlp[split] = {

            "embeddings": np.load(
                folder / f"{split}_embeddings.npy"
            ),

            "metadata": pd.read_csv(
                folder / f"{split}_metadata.csv"
            ),

        }

    return nlp

In [8]:
biobert = load_nlp_embeddings(
    BIOBERT_DIR
)

clinicalbert = load_nlp_embeddings(
    CLINICAL_DIR
)

In [9]:
for split in [

    "train",

    "validation",

    "test",

]:

    print("=" * 60)

    print(split.upper())

    print(

        "BioBERT :",

        biobert[split]["embeddings"].shape,

    )

    print(

        "ClinicalBERT :",

        clinicalbert[split]["embeddings"].shape,

    )

TRAIN
BioBERT : (257, 768)
ClinicalBERT : (257, 768)
VALIDATION
BioBERT : (56, 768)
ClinicalBERT : (56, 768)
TEST
BioBERT : (56, 768)
ClinicalBERT : (56, 768)


# Patient Alignment

The fusion model assumes that every image feature corresponds to the
correct radiology report.

Therefore, alignment is performed using the patient ID rather than the
row order.

In [10]:
def verify_alignment(cv_data, nlp_data, model_name):
    """
    Verify patient alignment between CV and NLP splits.

    🔧 FIX: the original version compared cv_ids == nlp_ids directly.
    If both files contain the same patients in a different row order
    (which is NOT guaranteed between Notebook 10 outputs and NLP
    embedding outputs — see cv_module/dataset.py warning), this assert
    would fail even though there is no real alignment problem, OR it
    could silently pass while rows are mismatched if the files happen
    to share a common but wrong order. 

    This version always sorts both id lists before comparing membership,
    then re-sorts the actual arrays by patient_id before returning them
    so downstream build_fusion_dataset() always concatenates in the
    same canonical order (lexicographic by patient_id).
    """

    print("=" * 70)
    print(model_name.upper())
    print("=" * 70)

    for split in ["train", "validation", "test"]:

        cv_ids  = cv_data[split]["metadata"]["patient_id"].tolist()
        nlp_ids = nlp_data[split]["metadata"]["patient_id"].tolist()

        print(f"\n{split.upper()}")
        print("CV Patients :", len(cv_ids))
        print("NLP Patients:", len(nlp_ids))
        print("Same Patients:", sorted(cv_ids) == sorted(nlp_ids))  # 🔧 set equality, not order

        assert set(cv_ids) == set(nlp_ids), (
            f"{split} alignment failed — patient sets do not match."
        )

        if cv_ids != nlp_ids:
            print(f"  ⚠️  Row order differs in {split} — will be re-sorted by patient_id in build_fusion_dataset().")
        else:
            print("  ✓  Row order matches.")


In [11]:
verify_alignment(

    cv,

    biobert,

    "BioBERT",

)

verify_alignment(

    cv,

    clinicalbert,

    "ClinicalBERT",

)

BIOBERT

TRAIN
CV Patients : 257
NLP Patients: 257
Same Patients: True
  ✓  Row order matches.

VALIDATION
CV Patients : 56
NLP Patients: 56
Same Patients: True
  ✓  Row order matches.

TEST
CV Patients : 56
NLP Patients: 56
Same Patients: True
  ✓  Row order matches.
CLINICALBERT

TRAIN
CV Patients : 257
NLP Patients: 257
Same Patients: True
  ✓  Row order matches.

VALIDATION
CV Patients : 56
NLP Patients: 56
Same Patients: True
  ✓  Row order matches.

TEST
CV Patients : 56
NLP Patients: 56
Same Patients: True
  ✓  Row order matches.


# Build Fusion Dataset

Combine image features, text embeddings, and patient identifiers into a
single multimodal dataset.

This step does not perform any learning. It only prepares the data for
the Fusion Encoder.

In [12]:
def build_fusion_dataset(cv_data, nlp_data):
    """
    Combine image features, text embeddings, and patient identifiers.

    🔧 FIX: both arrays are now sorted by patient_id before concatenation.
    Row order in Notebook 10's bottleneck_features_{split}.npy is not
    guaranteed to match row order in the NLP {split}_embeddings.npy.
    Sorting by patient_id canonicalizes both arrays so row i in
    image_features always corresponds to row i in text_features.
    """

    fusion = {}

    for split in ["train", "validation", "test"]:

        cv_meta  = cv_data[split]["metadata"].copy().sort_values("patient_id").reset_index(drop=True)
        nlp_meta = nlp_data[split]["metadata"].copy().sort_values("patient_id").reset_index(drop=True)

        # Re-order feature arrays to match the sorted metadata
        cv_order  = cv_data[split]["metadata"]["patient_id"].argsort().values  
        nlp_order = nlp_data[split]["metadata"]["patient_id"].argsort().values

        # Use sort_values index mapping instead
        cv_sort_idx  = cv_data[split]["metadata"].sort_values("patient_id").index.values
        nlp_sort_idx = nlp_data[split]["metadata"].sort_values("patient_id").index.values

        fusion[split] = {
            "image_features": cv_data[split]["features"][cv_sort_idx],
            "text_features":  nlp_data[split]["embeddings"][nlp_sort_idx],
            "metadata":       cv_meta,   # sorted, reset index
        }

    return fusion


In [13]:
fusion_biobert = build_fusion_dataset(

    cv,

    biobert,

)

fusion_clinicalbert = build_fusion_dataset(

    cv,

    clinicalbert,

)

In [14]:
for split in fusion_biobert:

    print("=" * 60)

    print(split.upper())

    print(

        "Image :",

        fusion_biobert[split]["image_features"].shape,

    )

    print(

        "Text :",

        fusion_biobert[split]["text_features"].shape,

    )

    print(

        "Metadata :",

        fusion_biobert[split]["metadata"].shape,

    )

TRAIN
Image : (257, 256)
Text : (257, 768)
Metadata : (257, 4)
VALIDATION
Image : (56, 256)
Text : (56, 768)
Metadata : (56, 4)
TEST
Image : (56, 256)
Text : (56, 768)
Metadata : (56, 4)


# Save Fusion Dataset

Store the prepared multimodal datasets for future Fusion notebooks.

In [15]:
def save_fusion_dataset(

    fusion_data,

    model_name,

):

    save_dir = OUTPUT_DIR / model_name

    save_dir.mkdir(

        parents=True,

        exist_ok=True,

    )

    for split in [

        "train",

        "validation",

        "test",

    ]:

        np.savez_compressed(

            save_dir /
            f"{split}_fusion.npz",

            image_features=fusion_data[split]["image_features"],

            text_features=fusion_data[split]["text_features"],

        )

        fusion_data[split]["metadata"].to_csv(

            save_dir /
            f"{split}_metadata.csv",

            index=False,

        )

    print(
        f"{model_name} saved successfully."
    )

In [16]:
save_fusion_dataset(

    fusion_biobert,

    "biobert",

)

save_fusion_dataset(

    fusion_clinicalbert,

    "clinicalbert",

)

biobert saved successfully.
clinicalbert saved successfully.


# Verification

Reload the saved multimodal datasets and verify their integrity.

In [17]:
for model in [

    "biobert",

    "clinicalbert",

]:

    print("=" * 70)

    print(model.upper())

    folder = OUTPUT_DIR / model

    for split in [

        "train",

        "validation",

        "test",

    ]:

        data = np.load(

            folder /
            f"{split}_fusion.npz"

        )

        metadata = pd.read_csv(

            folder /
            f"{split}_metadata.csv"

        )

        print(

            f"{split:12}",

            data["image_features"].shape,

            data["text_features"].shape,

            metadata.shape,

        )

BIOBERT
train        (257, 256) (257, 768) (257, 4)
validation   (56, 256) (56, 768) (56, 4)
test         (56, 256) (56, 768) (56, 4)
CLINICALBERT
train        (257, 256) (257, 768) (257, 4)
validation   (56, 256) (56, 768) (56, 4)
test         (56, 256) (56, 768) (56, 4)


# Save Fusion Dataset

Save each split as a compressed multimodal dataset.

Each file contains:

- Image Features
- Text Features
- Patient IDs

Metadata is stored separately for analysis and reporting.

In [18]:
print("=" * 70)

print("Fusion Dataset Preparation Completed Successfully")

print("=" * 70)

for model in [

    "biobert",

    "clinicalbert",

]:

    print(f"\n{model.upper()}")

    folder = OUTPUT_DIR / model

    for split in [

        "train",

        "validation",

        "test",

    ]:

        print(

            f"{split:12}",

            "✓",

            folder /
            f"{split}_fusion.npz",

        )

print("\nOutput Directory:")

print(OUTPUT_DIR)

Fusion Dataset Preparation Completed Successfully

BIOBERT
train        ✓ /kaggle/working/datasets/processed/fusion/biobert/train_fusion.npz
validation   ✓ /kaggle/working/datasets/processed/fusion/biobert/validation_fusion.npz
test         ✓ /kaggle/working/datasets/processed/fusion/biobert/test_fusion.npz

CLINICALBERT
train        ✓ /kaggle/working/datasets/processed/fusion/clinicalbert/train_fusion.npz
validation   ✓ /kaggle/working/datasets/processed/fusion/clinicalbert/validation_fusion.npz
test         ✓ /kaggle/working/datasets/processed/fusion/clinicalbert/test_fusion.npz

Output Directory:
/kaggle/working/datasets/processed/fusion
